In [396]:
## read in data and libraries
import pandas as pd
import numpy as np
from mcts import mcts
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import log_loss
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

In [397]:
game_data = pd.read_csv('/Users/maryellenfaulconer/Documents/Basketball_Project/full_data.csv')

In [398]:
latest_team_stats = (
    game_data
    .sort_values("date")
    .groupby("team")
    .tail(1)
)

latest_team_stats = latest_team_stats[
    [
        "team",
        "avg_points_scored",
        "avg_points_conceded",
        "std_points_scored",
        "std_points_conceded"
    ]
]

In [399]:
## quick data check
game_data.head()

,game_id,date,team,first_half_score,second_half_score,home_away,points_scored,points_conceded,win_loss,avg_points_scored,...,kenpom_available,kenpom_rank_diff,NetRtg_diff,ORtg_diff,DRtg_diff,Luck_diff,avg_points_scored_diff,std_points_scored_diff,avg_points_conceded_diff,std_points_conceded_diff
0,401823449,2025-11-03T13:00Z,Winthrop Eagles,40.0,41.0,1,81.0,74.0,1,74.416142,...,0,-181.0,0.00,-115.8,-117.2,-0.067,0.0,0.0,0.0,0.0
1,401823449,2025-11-03T13:00Z,Queens University Royals,35.0,39.0,0,74.0,81.0,0,74.416142,...,1,0.0,-1.44,0.0,0.0,0.000,0.0,0.0,0.0,0.0
2,401817194,2025-11-03T16:00Z,Bradley Braves,26.0,37.0,0,63.0,69.0,0,74.416142,...,0,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.0
3,401817194,2025-11-03T16:00Z,St. Bonaventure Bonnies,34.0,35.0,1,69.0,63.0,1,74.416142,...,0,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.0
4,401830655,2025-11-03T16:30Z,East Texas A&M Lions,68.0,51.0,1,119.0,60.0,1,74.416142,...,0,0.0,0.00,0.0,0.0,0.000,0.0,0.0,0.0,0.0


In [400]:
game_data = game_data[game_data["kenpom_available"] == 1]

In [401]:
## randomy choose one row per game, then drop the other row for that game
game_data = game_data.groupby("game_id").sample(1, random_state=42)

In [402]:
## train test split - time based split to avoid data leakage. use 80% of the data for training and 20% for testing, with the most recent games in the test set
game_data = game_data.sort_values("date")

# 2. Define features (REMOVE win_loss)
feature_cols = [
    "kenpom_rank_diff",
    "NetRtg_diff",
    "ORtg_diff",
    "DRtg_diff",
    "Luck_diff",
    "avg_points_scored_diff",
    "avg_points_conceded_diff",
    "std_points_scored_diff",
    "std_points_conceded_diff"
]

# 3. Define target
y = game_data["win_loss"]

# 4. Time-based split
split_idx = int(0.8 * len(game_data))

train = game_data.iloc[:split_idx]
test  = game_data.iloc[split_idx:]

# 5. Split features + target
X_train = train[feature_cols]
y_train = train["win_loss"]

X_test = test[feature_cols]
y_test = test["win_loss"]

In [403]:
## set up and train model monte carlo tree search model
model1 = RandomForestClassifier(n_estimators=100, random_state=42)
model1.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [404]:
## assess model performance
y_pred = model1.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.700507614213198


In [405]:
y_prob = model1.predict_proba(X_test)[:, 1]
print("Log Loss:", log_loss(y_test, y_prob))

Log Loss: 0.8967946217924953


In [406]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    param_grid,
                    cv=3,
                    scoring="accuracy")

grid.fit(X_train, y_train)

model2 = grid.best_estimator_

In [407]:
## print best hyperparameters
print("Best Hyperparameters:", grid.best_params_)

Best Hyperparameters: {'max_depth': 5, 'min_samples_split': 10, 'n_estimators': 100}


In [408]:
## fit model with best hyperparameters
model2.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, min_samples_split=10, random_state=42)

In [409]:
## assess model performance
y_pred = model2.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.733502538071066


In [410]:
## use fitted model to predict with monte carlo tree search
def create_matchup(team_a, team_b, team_stats):
    a = team_stats.loc[team_a]
    b = team_stats.loc[team_b]

    row = {}

    row["kenpom_rank_diff"] = a["kenpom_rank"] - b["kenpom_rank"]
    row["NetRtg_diff"] = a["NetRtg"] - b["NetRtg"]
    row["ORtg_diff"] = a["ORtg"] - b["ORtg"]
    row["DRtg_diff"] = a["DRtg"] - b["DRtg"]
    row["Luck_diff"] = a["Luck"] - b["Luck"]

    row["avg_points_scored_diff"] = a["avg_points_scored"] - b["avg_points_scored"]
    row["avg_points_conceded_diff"] = a["avg_points_conceded"] - b["avg_points_conceded"]

    row["std_points_scored_diff"] = a["std_points_scored"] - b["std_points_scored"]
    row["std_points_conceded_diff"] = a["std_points_conceded"] - b["std_points_conceded"]

    return pd.DataFrame([row])[feature_cols] 

In [411]:
## STEP 2: Single game simulation to use in monte carlo tree search
def simulate_game(team_a, team_b, model, team_stats):
    X_ab = create_matchup(team_a, team_b, team_stats)
    X_ba = create_matchup(team_b, team_a, team_stats)

    p_ab = model.predict_proba(X_ab)[0][1]
    p_ba = model.predict_proba(X_ba)[0][1]

    prob = (p_ab + (1 - p_ba)) / 2

    result = np.random.rand() < prob
    return team_a if result else team_b

In [412]:
## STEP 3: Monte Carlo Tree Search for tournament simulation
def simulate_series(team_a, team_b, model, team_stats, n=1000):
    wins = 0

    for _ in range(n):
        winner = simulate_game(team_a, team_b, model, team_stats)
        if winner == team_a:
            wins += 1

    prob = wins / n

    print(f"{team_a} win probability: {prob:.3f}")
    print(f"{team_b} win probability: {1 - prob:.3f}")

    return prob

In [413]:
team_stats = pd.read_csv('/Users/maryellenfaulconer/Documents/Basketball_Project/March_Madness.csv')

## rename Team Name to team for merging
team_stats = team_stats.rename(columns={"Team": "kenpom_rank"})
team_stats = team_stats.rename(columns={"Team Name": "Team"})
latest_team_stats = latest_team_stats.rename(columns={"team": "Team"})
team_stats = team_stats.rename(columns={"Luck ": "Luck"})

In [414]:
print(team_stats.columns)

Index(['Team', 'kenpom_rank', 'NetRtg', 'ORtg', 'DRtg', 'Luck'], dtype='object')


In [415]:
print(latest_team_stats.columns)

Index(['Team', 'avg_points_scored', 'avg_points_conceded', 'std_points_scored',
       'std_points_conceded'],
      dtype='object')


In [416]:
team_stats = latest_team_stats.merge(
    team_stats,
    on="Team",
    how="inner"   # only keep teams with KenPom (important)
)

In [417]:
## change index to team for easier lookup in create_matchup function
team_stats = team_stats.set_index("Team")

In [418]:
team_stats.head()

,avg_points_scored,avg_points_conceded,std_points_scored,std_points_conceded,kenpom_rank,NetRtg,ORtg,DRtg,Luck
Team,,,,,,,,,
Tennessee State Tigers,79.516129,73.096774,12.792891,13.476782,187,-1.83,109.1,110.9,0.070
High Point Panthers,89.484848,69.454545,15.004419,13.240777,92,8.40,117.0,108.6,0.048
Northern Iowa Panthers,69.000000,60.411765,11.964594,10.418857,72,11.81,110.0,98.2,-0.070
Queens University Royals,84.242424,82.212121,10.574233,13.338754,181,-1.44,115.8,117.2,0.067
North Dakota State Bison,79.636364,68.454545,13.251930,11.597658,113,5.13,111.7,106.6,0.040


In [521]:
simulate_series("Arizona Wildcats", "Duke Blue Devils", model2, team_stats, n=2000)

Arizona Wildcats win probability: 0.449
Duke Blue Devils win probability: 0.551


0.4485

Why did I not use logistic regression for classification?

1. I already did random forest and it worked. I wanted to build a regression to compare. since forest was having a weird time with the weight of which team is listed first, but ran into some issues with scaling
2. Issues with scaling was returning 100% rates which I dont like

If a lower seed had a win prob of .4 or more I chose them to spice things up. Its called MaryEllen's spicy hyperparameter

Interesting Probs:
VCU had about .4-.42 against NC Tar Heels
Iowa way over Clemson
Utah State way over Villanova
BYU 65% over Texas
Georgia only 63% over St Louis
Santa Clara Broncos (59%) over Kentucky (Rounf of 64)
Louisville Cardinals (54%) over Michigan State (Round of 32)
Arkansas supposed to destroy wisconsin (Round of 32)
Gonzaga (71%) over BYU (Round of 32)
Louisville Cardinals (57%) over UConn Huskies (Sweet 16)
Illinois (.497) against Houston (Sweet 16) (Advance due to MaryEllen hyperparameter)
Louisville run comes to end at Elite Eight (.2 against Duke)
Illinois (.485) against Florida (Elite 8) (Advancce due to MaryEllen hyperparameter)
Arizona (.515) barely beat Michigan (Final 4)
Duke wins it all (.547) over Arizona

"Upsets" between model + MaryEllen Rule:
Round of 64 - 5
Round of 32 - 3
Sweet 16 - 2
Elite 8 - 1
Final 4 - 0
Champs - NA

